# Week 7 — Feature Selection
## Global Life Expectancy Paradox

**Aim:** rank candidate country-level indicators by their individual statistical association with average life expectancy, then select the top eight for an interpretable reduced feature set.

**Method:** `SelectKBest` with `f_regression`. For each predictor, the F-test measures evidence of a linear association with the continuous target. The score is a ranking aid; it does not establish causation or show that a feature adds unique information after controlling for other variables.

**Leakage precaution:** male and female life expectancy are used to calculate the target, so they are not candidate predictors. Precomputed scaled/log columns are also excluded to avoid duplicate representations.

In [ ]:
from google.colab import files
import io
uploaded = files.upload()  # Upload the supplied feature-engineered CSV.
filename = next(iter(uploaded))
import pandas as pd, numpy as np
df_all = pd.read_csv(io.BytesIO(uploaded[filename]))
print("Uploaded shape:", df_all.shape)
display(df_all.head())

## Step 1 — Prepare the modelling columns
Keep identifiers for display, but use only the original numeric indicators as predictors.

In [ ]:
# Use only the original cleaned columns. Do not use precomputed *_scaled or *_minmax
# columns: doing so would duplicate information and distort PCA.
base_cols = [
    "name","region","iso2","population","pop_density","life_expectancy_male",
    "life_expectancy_female","sex_ratio","gdp_per_capita","gdp_growth",
    "unemployment","infant_mortality","fertility","homicide_rate",
    "urban_population","internet_users","secondary_school_enrollment_female",
    "secondary_school_enrollment_male","co2_emissions","forested_area"
]
df = df_all[base_cols].copy()
df["avg_life_expectancy"] = (df["life_expectancy_male"] + df["life_expectancy_female"]) / 2
target = "avg_life_expectancy"
predictors = [
    "population","pop_density","sex_ratio","gdp_per_capita","gdp_growth","unemployment",
    "infant_mortality","fertility","homicide_rate","urban_population","internet_users",
    "secondary_school_enrollment_female","secondary_school_enrollment_male",
    "co2_emissions","forested_area"
]
X = df[predictors].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median(numeric_only=True))
y = df[target].astype(float)
print(f"Rows: {len(df)} | candidate predictor features: {len(predictors)}")

## Step 2 — Score and select features
The top eight predictors are selected by F-score. Correlation sign is included to make the direction of association easier to interpret.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
selector = SelectKBest(score_func=f_regression, k=8)
selector.fit(X, y)
f_scores, p_values = f_regression(X, y)
results = pd.DataFrame({
    "feature": predictors,
    "F_score": f_scores,
    "p_value": p_values,
    "correlation_with_target": [X[c].corr(y) for c in predictors],
    "selected": selector.get_support()
}).sort_values("F_score", ascending=False)
display(results)
selected_features = results.loc[results["selected"], "feature"].tolist()
print("Selected features:", selected_features)

## Step 3 — Visualize feature scores
The bar chart shows relative F-scores; larger scores indicate stronger univariate linear association with the target in this dataset.

In [ ]:
import matplotlib.pyplot as plt
plot_data = results.sort_values("F_score")
plt.figure(figsize=(9, 6))
plt.barh(plot_data["feature"], plot_data["F_score"])
plt.xlabel("F-score"); plt.title("Univariate Feature Selection Scores")
plt.tight_layout(); plt.show()

## Step 4 — Export results
The results file documents the ranking and selected status of each candidate feature.

In [ ]:
results.to_csv("feature_selection_results.csv", index=False)
from google.colab import files
files.download("feature_selection_results.csv")